# 레슨 18 (후속): <em>사람</em>이 행동을 승인했다는 증거가 되는 영수증  

이 레슨은 <strong>에이전트</strong>가 수행한 작업과 <strong>게이트</strong>가 결정한 내용을 증명합니다. 이 노트북은 누락된 절반, 즉 **지정된 사람이** <strong>정확한</strong> 행동을 승인했다는 증거 — 완전한 정규화된 행동에 대해 별도로 사람이 보유한 서명(오프라인에서 검증된)을 추가합니다.  

여기 두 아티팩트 모두 레슨의 영수증과 <strong>동일한 봉투 형식</strong>을 사용합니다: `type` 필드를 가진 평면 페이로드이며, 정규화된 JCS 바이트에 대해 Ed25519로 직접 서명하고 구조화된 `signature` 객체가 첨부(서명된 바이트에는 제외)되어 있습니다. 승인 영수증은 액션 타입과 함께 새로운 `type`(`human.approval.v1`)으로, 하나의 `verify_chain`이 메인 노트북에서 만든 동일한 코드 경로를 통해 두 아티팩트 유형을 모두 처리합니다. 이 인간 승인 영수증은 교육용 구성으로 여기서 정의되었으며, draft-farley-acta-signed-receipts가 정의한 영수증 유형이 아닙니다.  

메인 노트북 데모 검증기에서 의도적인 업그레이드 하나: 여기 검증기는 영수증에 포함된 공개 키를 신뢰하는 대신 <strong>고정된 키 레지스트리</strong>에 대해 `signature.key_id`를 조회합니다. 이는 레슨의 체크리스트에서 권장하는 배포 환경 자세("검증용 공개 키를 게시하기")와 같으며, 위조가 자기 키를 가져오는 우회가 아니라 거부로 작동하게 하는 핵심입니다.  

이 노트북이 가르치는 규칙: **서명된 승인은 그 자체로 권한이 아닙니다.** 권한은 승인 영수증과 행동 영수증이 실행 시점에 여전히 동일한 정규화된 행동에 묶여 있고, 정책 버전, 키, 만료가 현재 상태이며, 승인이 아직 소비되지 않은 경우에만 존재합니다. 모든 실패는 <strong>별개의 이유</strong>로 거부하므로 <em>권한이 만료된 경우</em>와 <em>실행된 행동이 변경된 경우</em>를 구분할 수 있습니다.  


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## 정확한 동작

승인 단위는 <strong>정확한 동작 객체</strong>입니다 — "환불 승인" 같은 모호한 레이블이 아니라, 정확하고 완전히 명시된 동작입니다. 전체 객체에 서명하고(거기서 다이제스트를 추출하는 것) 나중에 사람이 <em>이 동작</em>만 승인했음을 증명할 수 있게 해줍니다.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## 하나의 봉투, 두 개의 권한

모든 영수증은 수업의 봉투입니다: `type` 필드가 있는 단평 페이로드와 서명된 바이트에 포함되지 않는 `signature` 객체(`alg`, `sig`, `key_id`)가 있습니다. `verify_envelope`는 두 종류의 영수증 모두에 대한 공통 구조 + 서명 확인이며; `signature.key_id`를 어느 <strong>고정된 키 레지스트리</strong>로 확인하는지가 권한을 구분합니다:

- **승인 영수증** (`human.approval.v1`) — 명시된 승인자, 전체 정규화된 작업 **및 그 다이제스트**, `policy_version`, 발행 및 만료 타임스탬프. 일회성 소비는 체인 수준에서 추적됩니다.
- **작업 영수증** (`agent.action.v1`) — 에이전트 신원, `run_id`, 동일한 정규화된 작업 <strong>다이제스트</strong>, 실행 결과 및 타임스탬프, 그리고 `parent_approval_ref`: 승인 영수증 해시(`receipt_hash`)로, 수업 체인의 `previous_receipt_hash`와 동일한 관례입니다.

공통 `action_digest` 필드가 결합하는 연결 고리입니다. `key_id`는 조회 힌트로서 서명 객체에 있지만: 이를 다른 고정된 키로 변경하면 서명 확인이 실패하므로 아무 효과가 없습니다.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: 권한 부여가 실제로 결정되는 곳

`verify_chain`은 두 개의 서명 검사를 위한 편의 래퍼가 <strong>아닙니다</strong>. 공유된 표준 `action_digest`, 정책/키/만료의 <strong>신선도</strong>, 그리고 승인에 대한 <strong>일회성 사용</strong>이 함께 점검되는 유일한 장소로, 이는 *지금 당장* 실행되는 작업에 대해 검사됩니다.

각 실패는 <strong>별개의 이유</strong>를 통해 거부되므로, 거부를 접한 사용자는 권한이 만료되었는지(정책 이동, 키 교체, 승인 만료, 승인 소비) 아니면 아직 유효한 승인 하에서 실행된 작업이 변경되었는지(다이제스트 대체) 구분할 수 있습니다.


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## 바인딩이 포착하는 것

아래의 모든 경우는 <strong>고유한 이유</strong>로 <strong>닫힌</strong> 상태에서 실패합니다. 첫 번째 블록은 클래식 세트(변조, 혼란스러운 대리인, 재생 공격, 어느 권한에서든 위조, 잘못된 입력)입니다. 두 번째 블록은 속성이 단순히 주장된 것이 아니라 실제임을 만드는 쌍입니다:

- **오래된 권한** — 서명은 여전히 유효하지만 정책 버전이 변경되었거나, 승인자 키가 고정된 레지스트리에서 회전되었거나, 실행 전에 승인이 만료됨;
- **다이제스트 대체** — `parent_approval_ref`가 <em>실제</em> 승인으로 가리키는 유효하게 서명된 작업 영수증이지만, 그 승인에 대한 표준 작업 다이제스트가 실제 실행 중인 작업과 일치하지 않음.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## 이것이 증명하는 것 — 그리고 증명하지 않는 것

**증명하는 것:** 명명된 사람이 *이 정확한 정형화된 작업*(전체 작업 + 다이제스트, 고정된 레지스트리에서 확인된 키로 서명됨)을 승인했으며, 에이전트가 *정확히 그 승인된 작업*(동일한 다이제스트, 승인에 `receipt_hash`로 묶인 영수증, 레슨 자체 체인 관례)을 실행했다 — 승인 정책 버전, 키, 만료가 여전히 유효한 상태에서 딱 한 번. 어느 한쪽이라도 변경되면 체인은 닫히고, 거부 사유는 <strong>어느</strong> 속성이 깨졌는지 알려준다: 오래된 권한 대 변경된 작업.

**증명하지 않는 것:** 승인 UI가 인간에게 그들이 서명하려던 것을 보여줬다는 것(WYSIWYS는 별도의 문제이다), 키가 회전 전에 강제되거나 도난당하지 않았다는 것, 또는 후속 효과가 작업과 일치했다는 것. 서명됨 ≠ 승인됨: 오래된 정책, 회전된 키, 만료된 창, 또는 다른 다이제스트에 대한 유효한 서명은 여기서 아무런 권한을 부여하지 않는다.

두 가지 영수증 유형은 일부러 레슨의 봉투와 동일한 `verify_chain` 코드 경로를 공유한다: 메인 노트북에서 작업 영수증을 위해 만든 바인딩은 인간의 승인을 확인하는 같은 코드이다. 하나의 검증 계약, 별도의 고정 권한자들, 정형화된 작업 다이제스트와 아무 것도 아닌 것으로 연결됨.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
